In [ ]:
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
import h5py
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datetime import datetime

# ============================================================
# Config
# ============================================================

DATA_ROOT = "Training dataset"


RUN_TIME = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUT_DIR = os.path.join(
    "waveloss_mat_dataset_results",
    f"run_{RUN_TIME}"
)
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

EPOCHS = 1000
BATCH_SIZE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-5

WIDTH = 64
MODES_T = 96
MODES_X = 48
N_LAYERS = 4

IN_CHANNELS = 1
OUT_CHANNELS = 1

# ============================================================
# Dataset for multiple .mat files
#
# Folder structure:
#
# Training dataset/
#     1/
#         *.mat
#     2/
#         *.mat
#     ...
#     10/
#         *.mat
#
# Each .mat file contains:
# PT: [200, 501]
# PA: [200, 501]
# ============================================================

import glob
import random


# ============================================================
# Find all .mat files
# ============================================================

all_files = glob.glob(
    os.path.join(DATA_ROOT, "**", "*.mat"),
    recursive=True
)

all_files = sorted(all_files)

print("Total .mat samples found:", len(all_files))

assert len(all_files) > 0, \
    f"No .mat files found under: {DATA_ROOT}"

# You expect:
# 10 folders x 100 samples = 1000 samples

if len(all_files) != 1000:
    print(
        f"Warning: expected 1000 samples, "
        f"but found {len(all_files)}"
    )


# ============================================================
# Random split
# 80% train / 10% validation / 10% test
# ============================================================

random.seed(SEED)
random.shuffle(all_files)

n_total = len(all_files)

n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)

train_files = all_files[:n_train]

val_files = all_files[
    n_train:n_train + n_val
]

test_files = all_files[
    n_train + n_val:
]

print("\nDataset split:")
print("Train samples:", len(train_files))
print("Val samples  :", len(val_files))
print("Test samples :", len(test_files))


# ============================================================
# Compute normalization statistics
# using TRAINING DATA ONLY
# ============================================================

def compute_statistics(file_list):

    pt_sum = 0.0
    pt_sq_sum = 0.0

    pa_sum = 0.0
    pa_sq_sum = 0.0

    n_elements = 0

    print("\nCalculating normalization statistics...")

    for mat_path in tqdm(
        file_list,
        desc="Statistics",
        ncols=120
    ):
        with h5py.File(mat_path, "r") as f:
            PT = np.asarray(f["PT"], dtype=np.float64)
            PA = np.asarray(f["PA"], dtype=np.float64)
        
        # 统一为 [T, X] = [501, 200]
        if PT.shape == (200, 501):
            PT = PT.T
        if PA.shape == (200, 501):
            PA = PA.T
        
        assert PT.shape == (501, 200), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"
        
        assert PA.shape == (501, 200), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"

        pt_sum += PT.sum()
        pt_sq_sum += np.square(PT).sum()

        pa_sum += PA.sum()
        pa_sq_sum += np.square(PA).sum()

        n_elements += PT.size

    # mean
    pt_mean = pt_sum / n_elements
    pa_mean = pa_sum / n_elements

    # variance
    pt_var = (
        pt_sq_sum / n_elements
        - pt_mean ** 2
    )

    pa_var = (
        pa_sq_sum / n_elements
        - pa_mean ** 2
    )

    # avoid numerical negative values
    pt_var = max(pt_var, 0.0)
    pa_var = max(pa_var, 0.0)

    # std
    pt_std = np.sqrt(pt_var) + 1e-8
    pa_std = np.sqrt(pa_var) + 1e-8

    return (
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    )


pt_mean, pt_std, pa_mean, pa_std = \
    compute_statistics(train_files)

print("\nNormalization statistics:")
print(f"PT mean = {pt_mean:.6e}")
print(f"PT std  = {pt_std:.6e}")
print(f"PA mean = {pa_mean:.6e}")
print(f"PA std  = {pa_std:.6e}")


# ============================================================
# Dataset
# ============================================================

class MatHeatAcousticDataset(Dataset):

    def __init__(
        self,
        file_list,
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    ):

        self.file_list = file_list

        self.pt_mean = pt_mean
        self.pt_std = pt_std

        self.pa_mean = pa_mean
        self.pa_std = pa_std


    def __len__(self):

        return len(self.file_list)


    def __getitem__(self, idx):

        mat_path = self.file_list[idx]

        with h5py.File(mat_path, "r") as f:
            PT = np.asarray(f["PT"], dtype=np.float32)
            PA = np.asarray(f["PA"], dtype=np.float32)
        
        # ====================================================
        # Ensure [T, X] = [501, 200]
        # ====================================================
        if PT.shape == (200, 501):
            PT = PT.T
        
        if PA.shape == (200, 501):
            PA = PA.T
        
        assert PT.shape == (501, 200), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"
        
        assert PA.shape == (501, 200), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        # ====================================================
        # Normalize
        # ====================================================

        PT = (
            PT - self.pt_mean
        ) / self.pt_std

        PA = (
            PA - self.pa_mean
        ) / self.pa_std


        # ====================================================
        # [T, X] -> [C, T, X]
        #
        # [200,501]
        # ->
        # [1,200,501]
        # ====================================================

        PT = torch.tensor(
            PT,
            dtype=torch.float32
        ).unsqueeze(0)

        PA = torch.tensor(
            PA,
            dtype=torch.float32
        ).unsqueeze(0)

        return PT, PA


    def denormalize_pa(self, x):

        return (
            x * self.pa_std
            + self.pa_mean
        )


    def denormalize_pt(self, x):

        return (
            x * self.pt_std
            + self.pt_mean
        )


# ============================================================
# Create datasets
# ============================================================

train_dataset = MatHeatAcousticDataset(
    train_files,
    pt_mean,
    pt_std,
    pa_mean,
    pa_std
)

val_dataset = MatHeatAcousticDataset(
    val_files,
    pt_mean,
    pt_std,
    pa_mean,
    pa_std
)

test_dataset = MatHeatAcousticDataset(
    test_files,
    pt_mean,
    pt_std,
    pa_mean,
    pa_std
)


# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# Dataset check
# ============================================================

PT_batch, PA_batch = next(
    iter(train_loader)
)

print("\nBatch check:")
print("PT batch shape:", PT_batch.shape)
print("PA batch shape:", PA_batch.shape)


# ============================================================
# 2D Spectral Convolution
# input shape: B, C, T, X
# ============================================================

class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_t, modes_x):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes_t = modes_t
        self.modes_x = modes_x

        scale = 1 / (in_channels * out_channels)

        self.weights = nn.Parameter(
            scale * torch.randn(
                in_channels,
                out_channels,
                modes_t,
                modes_x,
                dtype=torch.cfloat
            )
        )

    def compl_mul2d(self, x, weights):
        return torch.einsum("bixy,ioxy->boxy", x, weights)

    def forward(self, x):
        B, C, T, X = x.shape

        x_ft = torch.fft.rfftn(x, dim=(-2, -1))

        out_ft = torch.zeros(
            B,
            self.out_channels,
            T,
            X // 2 + 1,
            device=x.device,
            dtype=torch.cfloat
        )

        mt = min(self.modes_t, T)
        mx = min(self.modes_x, X // 2 + 1)

        out_ft[:, :, :mt, :mx] = self.compl_mul2d(
            x_ft[:, :, :mt, :mx],
            self.weights[:, :, :mt, :mx]
        )

        x = torch.fft.irfftn(out_ft, s=(T, X), dim=(-2, -1))
        return x


# ============================================================
# FNO2D Model
# input : PT, shape [B, 1, T, X]
# output: PA, shape [B, 1, T, X]
# ============================================================

class FNO2D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        width=48,
        modes_t=32,
        modes_x=64,
        n_layers=4
    ):
        super().__init__()

        self.width = width

        # input channels: PT + t-grid + x-grid
        self.fc0 = nn.Conv2d(in_channels + 2, width, kernel_size=1)

        self.spectral_layers = nn.ModuleList()
        self.pointwise_layers = nn.ModuleList()

        for _ in range(n_layers):
            self.spectral_layers.append(
                SpectralConv2d(width, width, modes_t, modes_x)
            )
            self.pointwise_layers.append(
                nn.Conv2d(width, width, kernel_size=1)
            )

        self.fc1 = nn.Conv2d(width, 128, kernel_size=1)
        self.fc2 = nn.Conv2d(128, out_channels, kernel_size=1)

    def get_grid(self, shape, device):
        B, C, T, X = shape

        t = torch.linspace(0, 1, T, device=device)
        x = torch.linspace(0, 1, X, device=device)

        tt, xx = torch.meshgrid(t, x, indexing="ij")

        grid = torch.stack([tt, xx], dim=0)
        grid = grid.unsqueeze(0).repeat(B, 1, 1, 1)

        return grid

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat([x, grid], dim=1)

        x = self.fc0(x)

        for spec, pw in zip(self.spectral_layers, self.pointwise_layers):
            x1 = spec(x)
            x2 = pw(x)
            x = F.gelu(x1 + x2)

        x = F.gelu(self.fc1(x))
        x = self.fc2(x)

        return x


# ============================================================
# Wave-aware Loss
# ============================================================

def relative_l2(pred, target):
    """
    Relative L2 error over the whole batch.
    """
    return torch.norm(pred - target) / (
        torch.norm(target) + 1e-8
    )


def temporal_gradient(x):
    """
    First-order temporal derivative.
    x shape: [B, C, T, X]

    Output shape:
        [B, C, T-1, X]
    """
    return x[:, :, 1:, :] - x[:, :, :-1, :]


def spatial_gradient(x):
    """
    First-order spatial derivative.
    x shape: [B, C, T, X]

    Output shape:
        [B, C, T, X-1]
    """
    return x[:, :, :, 1:] - x[:, :, :, :-1]


def gradient_loss(pred, target):
    """
    Wavefront-aware gradient loss.

    Encourages the predicted PA field to reproduce
    temporal and spatial wavefront structures.
    """

    pred_dt = temporal_gradient(pred)
    target_dt = temporal_gradient(target)

    pred_dx = spatial_gradient(pred)
    target_dx = spatial_gradient(target)

    loss_t = F.l1_loss(
        pred_dt,
        target_dt
    )

    loss_x = F.l1_loss(
        pred_dx,
        target_dx
    )

    return loss_t, loss_x


def fft_loss(pred, target):
    """
    Frequency-domain loss.

    Compares the complex 2D Fourier spectra of
    predicted and ground-truth PA fields.
    """

    pred_fft = torch.fft.rfft2(
        pred,
        dim=(-2, -1),
        norm="ortho"
    )

    target_fft = torch.fft.rfft2(
        target,
        dim=(-2, -1),
        norm="ortho"
    )

    # Complex spectral difference
    diff = pred_fft - target_fft

    loss = torch.mean(
        torch.abs(diff)
    )

    return loss


def wave_aware_loss(
    pred,
    target,
    lambda_rel=0.1,
    lambda_t=0.1,
    lambda_x=0.1,
    lambda_fft=0.05
):
    """
    Composite wave-aware loss.

    L =
        MSE
        + lambda_rel  * Relative L2
        + lambda_t    * Temporal-gradient loss
        + lambda_x    * Spatial-gradient loss
        + lambda_fft  * Fourier-domain loss
    """

    # --------------------------------------------------------
    # Field reconstruction loss
    # --------------------------------------------------------

    mse = F.mse_loss(
        pred,
        target
    )

    # --------------------------------------------------------
    # Relative field error
    # --------------------------------------------------------

    rel = relative_l2(
        pred,
        target
    )

    # --------------------------------------------------------
    # Wavefront / gradient loss
    # --------------------------------------------------------

    grad_t, grad_x = gradient_loss(
        pred,
        target
    )

    # --------------------------------------------------------
    # Spectral loss
    # --------------------------------------------------------

    spectral = fft_loss(
        pred,
        target
    )

    # --------------------------------------------------------
    # Total loss
    # --------------------------------------------------------

    total = (
        mse
        + lambda_rel * rel
        + lambda_t * grad_t
        + lambda_x * grad_x
        + lambda_fft * spectral
    )

    return {
        "total": total,
        "mse": mse,
        "rel": rel,
        "grad_t": grad_t,
        "grad_x": grad_x,
        "fft": spectral
    }


# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):

    model.train()

    total_loss = 0.0
    total_mse = 0.0
    total_rel = 0.0

    total_grad_t = 0.0
    total_grad_x = 0.0
    total_fft = 0.0

    for heat, acoustic in loader:

        heat = heat.to(DEVICE)
        acoustic = acoustic.to(DEVICE)

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        pred = model(heat)

        # ----------------------------------------------------
        # Wave-aware loss
        # ----------------------------------------------------

        loss_dict = wave_aware_loss(
            pred,
            acoustic,
            lambda_rel=0.1,
            lambda_t=0.02,
            lambda_x=0.02,
            lambda_fft=0.0
        )

        loss = loss_dict["total"]

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_loss += loss.item()

        total_mse += (
            loss_dict["mse"].item()
        )

        total_rel += (
            loss_dict["rel"].item()
        )

        total_grad_t += (
            loss_dict["grad_t"].item()
        )

        total_grad_x += (
            loss_dict["grad_x"].item()
        )

        total_fft += (
            loss_dict["fft"].item()
        )

    N = len(loader)

    return (
        total_loss / N,
        total_mse / N,
        total_rel / N,
        total_grad_t / N,
        total_grad_x / N,
        total_fft / N
    )


@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    total_mse = 0.0
    total_mae = 0.0
    total_rel = 0.0

    total_grad_t = 0.0
    total_grad_x = 0.0
    total_fft = 0.0

    for heat, acoustic in loader:

        heat = heat.to(DEVICE)
        acoustic = acoustic.to(DEVICE)

        pred = model(heat)

        loss_dict = wave_aware_loss(
            pred,
            acoustic,
            lambda_rel=0.1,
            lambda_t=0.02,
            lambda_x=0.02,
            lambda_fft=0.0
        )

        mae = F.l1_loss(
            pred,
            acoustic
        )

        total_loss += (
            loss_dict["total"].item()
        )

        total_mse += (
            loss_dict["mse"].item()
        )

        total_mae += mae.item()

        total_rel += (
            loss_dict["rel"].item()
        )

        total_grad_t += (
            loss_dict["grad_t"].item()
        )

        total_grad_x += (
            loss_dict["grad_x"].item()
        )

        total_fft += (
            loss_dict["fft"].item()
        )

    N = len(loader)

    return (
        total_loss / N,
        total_mse / N,
        total_mae / N,
        total_rel / N,
        total_grad_t / N,
        total_grad_x / N,
        total_fft / N
    )


# ============================================================
# Plot functions
# ============================================================

def plot_loss(log_path):

    data = np.loadtxt(
        log_path,
        delimiter=",",
        skiprows=1
    )

    epoch = data[:, 0]

    # Correct column indices
    train_mse = data[:, 2]
    val_mse   = data[:, 8]

    train_rel = data[:, 3]
    val_rel   = data[:, 10]

    # --------------------------------------------------------
    # MSE
    # --------------------------------------------------------

    plt.figure(figsize=(5, 3.8))

    plt.semilogy(
        epoch,
        train_mse,
        label="Train MSE"
    )

    plt.semilogy(
        epoch,
        val_mse,
        label="Validation MSE"
    )

    plt.xlabel("Epoch")
    plt.ylabel("MSE")

    plt.legend(frameon=False)

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUT_DIR,
            "loss_curve.png"
        ),
        dpi=300
    )

    plt.close()

    # --------------------------------------------------------
    # Relative L2
    # --------------------------------------------------------

    plt.figure(figsize=(5, 3.8))

    plt.semilogy(
        epoch,
        train_rel,
        label="Train Rel. L2"
    )

    plt.semilogy(
        epoch,
        val_rel,
        label="Validation Rel. L2"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Relative L2")

    plt.legend(frameon=False)

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUT_DIR,
            "relative_l2_curve.png"
        ),
        dpi=300
    )

    plt.close()


@torch.no_grad()
def plot_prediction(model, dataset):
    model.eval()

    PT, PA = dataset[0]

    PT_gpu = PT.unsqueeze(0).to(DEVICE)
    pred = model(PT_gpu).cpu().squeeze(0).squeeze(0).numpy()

    PT = PT.squeeze(0).numpy()
    PA = PA.squeeze(0).numpy()

    PT_real = dataset.denormalize_pt(PT)
    PA_real = dataset.denormalize_pa(PA)
    pred_real = dataset.denormalize_pa(pred)

    err = pred_real - PA_real

    vmax = np.max(np.abs(PA_real))
    evmax = np.max(np.abs(err))

    plt.figure(figsize=(12, 3))

    plt.subplot(1, 4, 1)
    plt.imshow(PT_real, aspect="auto", cmap="inferno")
    plt.title("Input PT")
    plt.xlabel("x")
    plt.ylabel("t")
    plt.colorbar()

    plt.subplot(1, 4, 2)
    plt.imshow(PA_real, aspect="auto", cmap="seismic", vmin=-vmax, vmax=vmax)
    plt.title("GT PA")
    plt.xlabel("x")
    plt.ylabel("t")
    plt.colorbar()

    plt.subplot(1, 4, 3)
    plt.imshow(pred_real, aspect="auto", cmap="seismic", vmin=-vmax, vmax=vmax)
    plt.title("Pred PA")
    plt.xlabel("x")
    plt.ylabel("t")
    plt.colorbar()

    plt.subplot(1, 4, 4)
    plt.imshow(err, aspect="auto", cmap="seismic", vmin=-evmax, vmax=evmax)
    plt.title("Error")
    plt.xlabel("x")
    plt.ylabel("t")
    plt.colorbar()

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "prediction_comparison.png"), dpi=300)
    plt.close()

    np.savez(
        os.path.join(OUT_DIR, "prediction_result.npz"),
        PT=PT_real,
        PA=PA_real,
        PA_pred=pred_real,
        error=err
    )


# ============================================================
# Main training
# ============================================================

model = FNO2D(
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    width=WIDTH,
    modes_t=MODES_T,
    modes_x=MODES_X,
    n_layers=N_LAYERS
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

log_path = os.path.join(OUT_DIR, "training_log.csv")

with open(log_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch",
        
        "train_loss",
        "train_mse",
        "train_rel",
        "train_grad_t",
        "train_grad_x",
        "train_fft",
        
        "val_loss",
        "val_mse",
        "val_mae",
        "val_rel",
        "val_grad_t",
        "val_grad_x",
        "val_fft",
        
        "lr"
    ])

best_val = 1e9


epoch_bar = tqdm(range(1, EPOCHS + 1),
                 desc="Training",
                 ncols=120)

for epoch in epoch_bar:
    (
        train_loss,
        train_mse,
        train_rel,
        train_grad_t,
        train_grad_x,
        train_fft
    ) = train_one_epoch(
        model,
        train_loader,
        optimizer
    )
    
    
    (
        val_loss,
        val_mse,
        val_mae,
        val_rel,
        val_grad_t,
        val_grad_x,
        val_fft
    ) = evaluate(
        model,
        val_loader
    )

    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]

    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
        
            epoch,
        
            train_loss,
            train_mse,
            train_rel,
            train_grad_t,
            train_grad_x,
            train_fft,
        
            val_loss,
            val_mse,
            val_mae,
            val_rel,
            val_grad_t,
            val_grad_x,
            val_fft,
        
            lr_now
        ])

    if val_rel < best_val:

        best_val = val_rel
    
        torch.save(
            model.state_dict(),
            os.path.join(
                OUT_DIR,
                "best_fno2d.pt"
            )
        )
        
    if epoch % 100 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:04d} | "
            f"Train MSE {train_mse:.4e} | "
            f"Train Rel {train_rel:.4e} | "
            f"Val MSE {val_mse:.4e} | "
            f"Val Rel {val_rel:.4e}"
        )

    epoch_bar.set_postfix(
        train_mse=f"{train_mse:.2e}",
        val_mse=f"{val_mse:.2e}",
        rel=f"{val_rel:.2e}",
        lr=f"{lr_now:.1e}",
    )


# ============================================================
# Final test
# ============================================================

model.load_state_dict(
    torch.load(os.path.join(OUT_DIR, "best_fno2d.pt"), map_location=DEVICE)
)

(
    test_loss,
    test_mse,
    test_mae,
    test_rel,
    test_grad_t,
    test_grad_x,
    test_fft
) = evaluate(
    model,
    test_loader
)

print("\nFinal Test Results")
print(f"Test Total Loss : {test_loss:.6e}")
print(f"Test MSE        : {test_mse:.6e}")
print(f"Test MAE        : {test_mae:.6e}")
print(f"Test Rel L2     : {test_rel:.6e}")
print(f"Test Grad-t     : {test_grad_t:.6e}")
print(f"Test Grad-x     : {test_grad_x:.6e}")
print(f"Test FFT Loss   : {test_fft:.6e}")


with open(
    os.path.join(
        OUT_DIR,
        "test_results.txt"
    ),
    "w"
) as f:

    f.write("Final Test Results\n")
    f.write(f"Test Total Loss : {test_loss:.6e}\n")
    f.write(f"Test MSE        : {test_mse:.6e}\n")
    f.write(f"Test MAE        : {test_mae:.6e}\n")
    f.write(f"Test Rel L2     : {test_rel:.6e}\n")
    f.write(f"Test Grad-t     : {test_grad_t:.6e}\n")
    f.write(f"Test Grad-x     : {test_grad_x:.6e}\n")
    f.write(f"Test FFT Loss   : {test_fft:.6e}\n")


plot_loss(log_path)
plot_prediction(model, test_dataset)

print(f"\nAll results saved to: {OUT_DIR}")

Using device: cuda
Total .mat samples found: 800

Dataset split:
Train samples: 640
Val samples  : 80
Test samples : 80

Calculating normalization statistics...


Statistics:   0%|                                                                               | 0/640 [00:00…


Normalization statistics:
PT mean = 3.135567e+01
PT std  = 2.164740e+01
PA mean = -2.100887e+04
PA std  = 2.847824e+05

Batch check:
PT batch shape: torch.Size([8, 1, 501, 200])
PA batch shape: torch.Size([8, 1, 501, 200])


Training:   0%|                                                                                | 0/1000 [00:00…

In [1]:
!nvidia-smi

Wed Aug 19 17:05:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40-24Q                 On  |   00000000:02:01.0 Off |                    0 |
| N/A   N/A    P0            N/A  /  N/A  |   10937MiB /  24576MiB |     33%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!ps -fp 342272

UID          PID    PPID  C STIME TTY          TIME CMD
asusio    342272  342089 99 Aug17 ?        4-11:43:06 /opt/jupyterhub/bin/python


In [ ]:
!kill 563735